# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

1. `[4, 2]`. Для группы a складываются 1 и 3, а в группе b только 2.
2. `[1, 1] [2, 1]`. В группе a одно заполненное значение и две строки. В группе b одна строка с заполненным значением.
3. `20.0`. По умолчанию сводная таблица берёт среднее двух значений, 10 и 30.
4. `2 3`. При внутреннем соединении совпадают два id. Левое соединение сохраняет и третий id.
5. `4`. Для id 1 нашлись две строки справа. Ещё по одной строке остаётся для id 2 и 3.

In [1]:
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))

[4, 2]
[1, 1] [2, 1]
20.0
2 3
4


In [2]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [3]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
print("Форма после очистки", df.shape)

Форма после очистки (5280, 9)


In [4]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
print("Оплаченных заказов", len(paid), "| форма справочника", cu.shape)

Оплаченных заказов 4739 | форма справочника (2200, 4)


Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [5]:
by_city = df.groupby("city").agg(orders=("order_id", "count"), cancel_share=("cancel", "mean"),
                                known_days=("delivery_days", "count"), avg_days=("delivery_days", "mean"))
by_city["revenue"] = paid.groupby("city")["amount"].sum()
print(by_city.sort_values("revenue", ascending=False).round(3))

           orders  cancel_share  known_days  avg_days     revenue
city                                                             
Алматы       2116         0.078        2053     1.997  29144090.0
Астана       1339         0.088        1299     2.019  17884750.0
Шымкент       776         0.206         754     4.451   8602660.0
Караганда     519         0.091         495     1.941   7020200.0
Актобе        387         0.106         383     2.070   5117380.0
Атырау        143         0.077         138     2.036   1893850.0


**Вывод для менеджера.** В Шымкенте отменили 160 из 776 заказов (20,6%). Средний срок среди 754 заказов с известной доставкой — 4,45 дня. Это самые высокие значения среди шести городов. Стоит проверить, на каком этапе в Шымкенте увеличился срок доставки.

## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [6]:
rev = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="sum")
checks = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="mean")
counts = paid.pivot_table(index="category", columns="month", values="order_id", aggfunc="count")
rev["изменение, %"] = (rev[10] / rev[9] - 1) * 100
print("Выручка и изменение")
print(rev.round(2))
print("Средний чек")
print(checks.round(2))
print("Количество оплаченных заказов для каждого среднего")
print(counts)

Выручка и изменение
month                 9          10  изменение, %
category                                         
Дом           4997730.0   5125790.0          2.56
Книги         1065300.0   1140910.0          7.10
Красота       2610950.0   2678990.0          2.61
Одежда        9167100.0   8505730.0         -7.21
Электроника  17402330.0  16968100.0         -2.50
Средний чек
month              9         10
category                       
Дом          10499.43  10768.47
Книги         4020.00   3815.75
Красота       6187.09   6158.60
Одежда       13520.80  13045.60
Электроника  32772.75  33600.20
Количество оплаченных заказов для каждого среднего
month         9    10
category             
Дом          476  476
Книги        265  299
Красота      422  435
Одежда       678  652
Электроника  531  505


**Вывод для менеджера.** Результаты совпали с ДЗ 4. Электроника лидирует с выручкой 34 370 430 ₸ за два месяца, её средний чек в октябре — 33 600 ₸ по 505 оплатам. У одежды выручка упала на 7,21% (678 оплат в сентябре и 652 в октябре). Стоит проверить причины снижения по одежде. В ДЗ 4 построение одной таблицы выручки занимало 5 строк (список категорий, пустая матрица, цикл и две суммы), здесь — одну строку с pivot_table. В обоих случаях считаем без вывода и подготовки данных.

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [7]:
delivery = df.groupby("delivery_days").agg(orders=("order_id", "count"), cancelled=("cancel", "sum"),
                                           cancel_share=("cancel", "mean"))
print(delivery.round(4))
ratio = delivery.loc[8, "cancel_share"] / delivery.loc[1, "cancel_share"]
print("Отношение долей отмен для 8 дней и 1 дня", round(ratio, 2))
print("Заказов без известного срока", df["delivery_days"].isna().sum())

               orders  cancelled  cancel_share
delivery_days                                 
1.0              1557         66        0.0424
2.0              1570        125        0.0796
3.0              1590        213        0.1340
5.0                86         19        0.2209
6.0               120         26        0.2167
7.0                95         33        0.3474
8.0               104         45        0.4327
Отношение долей отмен для 8 дней и 1 дня 10.21
Заказов без известного срока 158


**Вывод для менеджера.** При сроке 8 дней отменили 45 из 104 заказов (43,3%), при сроке 1 день — 66 из 1 557 (4,2%). Разница в частоте отмен — в 10,21 раза. В Шымкенте в октябре сроки достигали 5–8 дней, поэтому там стоит проверить задержки и причины отмен. Сравнение по срокам показывает связь и само по себе не устанавливает причину.

## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [8]:
# Выручка складывается из оплаченных заказов. По умолчанию pivot_table вычисляет среднее.
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum")
p["изменение, %"] = (p[10] / p[9] - 1) * 100
print(p.round(2))
print("Оплаченных заказов по городам и месяцам")
print(paid.groupby(["city", "month"]).size().unstack())

month               9          10  изменение, %
city                                           
Актобе      2576960.0   2540420.0         -1.42
Алматы     14501070.0  14643020.0          0.98
Астана      9034220.0   8850530.0         -2.03
Атырау       923990.0    969860.0          4.96
Караганда   3648520.0   3371680.0         -7.59
Шымкент     4558650.0   4044010.0        -11.29
Оплаченных заказов по городам и месяцам
month       9    10
city               
Актобе     170  176
Алматы     959  993
Астана     603  618
Атырау      63   69
Караганда  250  222
Шымкент    327  289


**Вывод для менеджера.** Код из условия включил отменённые заказы и посчитал среднюю сумму вместо выручки. В Шымкенте выручка от 327 оплат в сентябре составила 4 558 650 ₸, а от 289 оплат в октябре — 4 044 010 ₸. Падение на 11,29% требует разбора причин отмен.

## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [9]:
m = df.merge(cu, on="customer_id", how="left")
print("До и после соединения", len(df), len(m), "| равны", len(df) == len(m))
print("Уникальный ключ клиентов", cu["customer_id"].is_unique, "| заказов без возраста", m["age"].isna().sum())
m["age_group"] = m["age"] // 10 * 10
age = m.groupby("age_group").agg(orders=("order_id", "count"), cancel_share=("cancel", "mean"))
paid_m = m[m["status"] == "оплачен"]
age["paid_orders"] = paid_m.groupby("age_group").size()
age["mean_check"] = paid_m.groupby("age_group")["amount"].mean()
age["median_check"] = paid_m.groupby("age_group")["amount"].median()
print(age.round(3))
print("Возраст в группе 60", sorted(m.loc[m["age_group"] == 60, "age"].unique().tolist()))

До и после соединения 5280 5280 | равны True
Уникальный ключ клиентов True | заказов без возраста 0
           orders  cancel_share  paid_orders  mean_check  median_check
age_group                                                             
10            229         0.087          209   14044.737       10700.0
20           1226         0.096         1108   14130.415       10100.0
30           1241         0.111         1103   15142.339       10670.0
40           1170         0.111         1040   14792.971       10050.0
50           1294         0.092         1175   14671.796       10240.0
60            120         0.133          104   16779.135       10495.0
Возраст в группе 60 [60]


**Вывод для менеджера.** Группа 60 состоит только из 60-летних. На 120 заказов пришлось 16 отмен (13,3%). Средний чек по 104 оплатам самый высокий (16 779 ₸), но медиана тех же оплат составляет 10 495 ₸. Например, у группы 30 медиана выше — 10 670 ₸ по 1 103 оплатам. Прежде чем запускать отдельную акцию, стоит проверить, сохраняется ли разница в следующие месяцы.

## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [10]:
per_client = df.groupby("customer_id").size()
print("Заказов на клиента → число клиентов")
print(per_client.value_counts().sort_index())
repeat = (per_client > 1).sum()
both = (df.groupby("customer_id")["month"].nunique() == 2).sum()
print(f"Повторно заказывали {repeat} из {len(per_client)} активных клиентов ({repeat / len(per_client):.1%}).")
print("Заказы в обоих месяцах", both)
print("Без заказов", len(cu) - len(per_client), "из", len(cu), "клиентов справочника")

Заказов на клиента → число клиентов
1     501
2     563
3     434
4     293
5     127
6      58
7      15
8       9
9       1
10      1
Name: count, dtype: int64
Повторно заказывали 1501 из 2002 активных клиентов (75.0%).
Заказы в обоих месяцах 1064
Без заказов 198 из 2200 клиентов справочника


**Вывод для менеджера.** Из 2 002 клиентов с заказами 1 501 (75,0%) оформил больше одного заказа, а 1 064 заказывали в обоих месяцах. Расчёт охватывает все 5 280 заказов, включая отменённые. Для кампании первой покупки есть отдельная аудитория — 198 клиентов справочника без единого заказа.

## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [11]:
sh = df[df["city"] == "Шымкент"]
sh_paid = sh[sh["status"] == "оплачен"]
months = sh.groupby("month").agg(orders=("order_id", "count"), cancel_share=("cancel", "mean"))
months["paid_orders"] = sh_paid.groupby("month").size()
months["mean_check"] = sh_paid.groupby("month")["amount"].mean()
months["revenue"] = sh_paid.groupby("month")["amount"].sum()
print(months.round(3))
expected = months.loc[10, "orders"] * (1 - months.loc[9, "cancel_share"]) * months.loc[10, "mean_check"]
loss = expected - months.loc[10, "revenue"]
company = paid.groupby("month")["amount"].sum()
company_drop = company[9] - company[10]
print(f"Шымкент в октябре по сценарию: {expected:,.0f} ₸. Разница с фактом: {loss:,.0f} ₸.")
print(f"Падение компании: {company_drop:,.0f} ₸. Сценарная разница покрывает {loss / company_drop:.1%} падения.")
print(f"Компания по сценарию: сентябрь {company[9]:,.0f} ₸, октябрь {company[10] + loss:,.0f} ₸.")

       orders  cancel_share  paid_orders  mean_check    revenue
month                                                          
9         359         0.089          327   13940.826  4558650.0
10        417         0.307          289   13993.114  4044010.0
Шымкент в октябре по сценарию: 5,315,006 ₸. Разница с фактом: 1,270,996 ₸.
Падение компании: 823,890 ₸. Сценарная разница покрывает 154.3% падения.
Компания по сценарию: сентябрь 35,243,410 ₸, октябрь 35,690,516 ₸.


**Вывод для менеджера.** В сентябре в Шымкенте отменили 32 из 359 заказов, в октябре — 128 из 417. Если при октябрьском потоке сохранить сентябрьскую долю отмен и октябрьский средний чек 13 993 ₸ (289 оплат), оценка недополученной выручки составит 1 270 996 ₸. Это больше фактического падения всей компании на 823 890 ₸. При этих допущениях компания выросла бы на 447 106 ₸. Сумма показывает масштаб возможного эффекта, фактическую причину отмен стоит проверить по заказам.

## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [12]:
channels = paid_m.pivot_table(index="channel", columns="month", values="amount", aggfunc="sum")
channels["изменение, %"] = (channels[10] / channels[9] - 1) * 100
print("Фактическая выручка каналов и её изменение")
print(channels.round(2))
print("Оплаченных заказов")
print(paid_m.groupby(["channel", "month"]).size().unstack())
print("Клиентов в каждой группе при каждом разбиении")
print(cu["channel"].value_counts())

# Фиксируем seed, чтобы 100 случайных разбиений можно было воспроизвести.
rng = np.random.default_rng(7)
changes = []
trial = paid.copy()
for i in range(100):
    # Выбираем все метки без возврата: численность трёх каналов сохраняется точно.
    labels = rng.choice(cu["channel"], size=len(cu), replace=False)
    # Одному клиенту назначается одна группа для всех его заказов в обоих месяцах.
    lookup = dict(zip(cu["customer_id"], labels))
    trial["group"] = trial["customer_id"].map(lookup)
    totals = trial.pivot_table(index="group", columns="month", values="amount", aggfunc="sum")
    change = (totals[10] / totals[9] - 1) * 100
    changes.append(change)

# Строка — одно разбиение, столбец — одна из трёх групп.
simulated = pd.DataFrame(changes)
print("Изменения выручки в первом разбиении, %")
print(simulated.iloc[0].round(2))
# percentile даёт границы центральных 90% результатов для группы размера органики.
low, high = np.percentile(simulated["органика"], [5, 95])
print(f"Органика по размеру: от {low:.1f}% до {high:.1f}% по 100 разбиениям.")

Фактическая выручка каналов и её изменение
month              9          10  изменение, %
channel                                       
органика  12212420.0  11484020.0         -5.96
реклама   18034980.0  17467600.0         -3.15
рефералы   4996010.0   5467900.0          9.45
Оплаченных заказов
month       9     10
channel             
органика   807   811
реклама   1213  1194
рефералы   352   362
Клиентов в каждой группе при каждом разбиении
channel
реклама     1133
органика     743
рефералы     324
Name: count, dtype: int64
Изменения выручки в первом разбиении, %
group
органика   -6.43
реклама    -2.02
рефералы    6.43
Name: 0, dtype: float64
Органика по размеру: от -9.9% до 6.2% по 100 разбиениям.


**Вывод для менеджера.** Органика потеряла 5,96% выручки при 807 оплатах в сентябре и 811 в октябре. В 100 случайных разбиениях диапазон между 5-м и 95-м процентилями для группы из 743 клиентов составил от −9,9% до +6,2%. Наблюдаемое падение внутри этого диапазона. Для решения по SEO стоит проверить динамику за несколько месяцев и данные поискового трафика.